# ROP Classification: Class-Balanced Training

クラス不均衡に対処するためのWeighted Cross-Entropy/Focal Lossを使用した学習。

## 変更点 (vs quality_comparison)
- **Weighted Cross-Entropy**: クラス頻度の逆数で重み付け
- **Weighted Focal Loss**: alpha（クラス重み）を追加
- **WeightedRandomSampler**: 少数クラスのオーバーサンプリング（オプション）

## データセット概要
- **Quality filter**: Good + Fair
- **総画像数**: 6,448枚（Good: 4,494 / Fair: 1,954）

## クラス不均衡の状況 (good_fair)
| Task | Class | Count | Ratio |
|------|-------|-------|-------|
| Zone | I / II / III | 651 / 3,127 / 1,200 | 13% / 63% / 24% |
| Stage | 0 / 1 / 2 / 3 | 1,969 / 1,355 / 823 / 809 | 40% / 27% / 17% / 16% |
| Aggressive ROP | No / Yes | 4,750 / 228 | 95% / 5% |
| Treatment | No / Yes | 4,351 / 627 | 87% / 13% |

## 再学習（リセット）する場合
学習済みモデルが存在するとスキップされるため、再学習するには以下を削除する:

```
outputs_class_balanced/          ← ディレクトリごと削除（推奨）
├── fold_1/best_model.pt         ← または各foldのモデルファイルのみ削除
├── fold_2/best_model.pt
├── fold_3/best_model.pt
├── fold_4/best_model.pt
├── fold_5/best_model.pt
├── predictions.csv              ← 全fold統合の予測結果
└── config.json                  ← 設定・結果サマリ
```

**スキップ判定**: `run_cross_validation()` 内で `best_model.pt` の存在を確認。
ファイルが存在する場合、そのfoldの学習をスキップして検証のみ実行する。

In [1]:
# ==================== Environment Setup ====================
import os
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Image processing
import cv2
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# timm for EfficientNet
import timm

# Metrics
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, classification_report,
    cohen_kappa_score, accuracy_score, f1_score
)

# Albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

from datetime import datetime

# Seed
def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Quadro RTX 5000


In [2]:
# ==================== Configuration ====================

class Config:
    """Training configuration with class balancing"""

    # Paths
    DATA_ROOT = Path(r"E:\Multicenter_ROP_study")
    KUBOTA_DIR = DATA_ROOT / "Multicenter_images" / "Kubota_selection"
    EXCEL_PATH = DATA_ROOT / "multicenter_patient_data_20260127.xlsx"
    OUTPUT_DIR = Path(r"C:\Users\ykita\ROP_AI_project\ROP_project\multicenter_study\outputs_class_balanced")

    # Model
    MODEL_NAME = "efficientnet_b0"
    PRETRAINED = True
    DROPOUT = 0.5

    # Image
    IMG_SIZE = 512

    # Training
    BATCH_SIZE = 16
    NUM_WORKERS = 0
    EPOCHS = 200
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-3
    PATIENCE = 15  # 少し長めに

    # Cross-validation
    N_FOLDS = 5

    # Task weights (タスク間の重み)
    TASK_WEIGHTS = {
        'zone': 1.0,
        'stage': 1.0,
        'plus': 1.0,
        'aggressive_rop': 1.5,
        'treatment': 1.5
    }

    # Label smoothing
    LABEL_SMOOTHING = 0.1  # 少し下げる

    # MixUp
    MIXUP_ALPHA = 0.2
    MIXUP_P = 0.5

    # Class balancing settings
    USE_CLASS_WEIGHTS = True      # Weighted CE
    USE_FOCAL_LOSS = True         # Focal Loss with alpha
    FOCAL_GAMMA = 2.0
    USE_WEIGHTED_SAMPLER = False  # まずはLoss側のみで試す

    # Quality filter
    QUALITY_FILTER = ["Good", "Fair"]  # good_fair条件

    # Version
    VERSION = "class_balanced_v1"

config = Config()
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {config.OUTPUT_DIR}")

Output directory: C:\Users\ykita\ROP_AI_project\ROP_project\multicenter_study\outputs_class_balanced


## 1. Data Loading

In [ ]:
# ==================== Load Data ====================

def load_quality_labeled_images(kubota_dir: Path) -> pd.DataFrame:
    """Load images from Kubota_selection folder."""
    data = []
    for quality in ["Good", "Fair", "Bad", "Worst"]:
        folder = kubota_dir / quality
        if not folder.exists():
            continue
        for img_path in folder.glob("*.png"):
            if img_path.stat().st_size == 0:
                continue
            filename = img_path.stem
            parts = filename.rsplit('_', 1)
            video_id = parts[0] if len(parts) >= 2 else filename
            data.append({
                "image_path": str(img_path),
                "video_id": video_id,
                "quality": quality
            })
    return pd.DataFrame(data)

def load_patient_data(excel_path: Path) -> pd.DataFrame:
    """Load and preprocess patient data."""
    df = pd.read_excel(excel_path)
    
    column_mapping = {
        'video_id': df.columns[0],
        'zone': df.columns[6],
        'stage': df.columns[7],
        'plus': df.columns[8],
        'aggressive_rop': df.columns[9],
        'treatment': df.columns[12],
    }
    rename_dict = {v: k for k, v in column_mapping.items()}
    df = df.rename(columns=rename_dict)
    
    # Encode labels
    df['zone_label'] = df['zone'].apply(lambda x: int(x) - 1 if pd.notna(x) and str(x).isdigit() else -1)
    df['stage_label'] = df['stage'].apply(lambda x: int(x) if pd.notna(x) and str(x).isdigit() else -1)
    
    # Plus label: Excel contains numeric values (0, 1, 2) not text
    def map_plus_label(x):
        if pd.isna(x):
            return -1
        # Handle numeric values directly
        if isinstance(x, (int, float)) and not np.isnan(x):
            val = int(x)
            if val in [0, 1, 2]:
                return val
        # Fallback: try text mapping
        text_map = {'normal': 0, 'preplus': 1, 'plus': 2, '0': 0, '1': 1, '2': 2}
        return text_map.get(str(x).lower().strip(), -1)
    
    df['plus_label'] = df['plus'].apply(map_plus_label)
    df['aggressive_rop_label'] = df['aggressive_rop'].apply(lambda x: 1 if str(x).lower().strip() in ['yes', 'y', '1', 'true'] else 0)
    df['treatment_label'] = df['treatment'].apply(lambda x: 1 if str(x).lower().strip() in ['yes', 'y', '1', 'true'] else 0)
    
    return df

# Load and merge
quality_df = load_quality_labeled_images(config.KUBOTA_DIR)
patient_df = load_patient_data(config.EXCEL_PATH)

dataset_df = quality_df.merge(
    patient_df[['video_id', 'zone_label', 'stage_label', 'plus_label', 'aggressive_rop_label', 'treatment_label']],
    on='video_id', how='left'
).dropna(subset=['zone_label'])

# Filter by quality
dataset_df = dataset_df[dataset_df['quality'].isin(config.QUALITY_FILTER)].copy()

print(f"Dataset size: {len(dataset_df)}")
print(f"Quality distribution: {dataset_df['quality'].value_counts().to_dict()}")

# Plus label distribution check
print(f"\nPlus label distribution:")
print(dataset_df['plus_label'].value_counts().sort_index())

In [ ]:
# ==================== Analyze Class Distribution ====================

def analyze_class_distribution(df: pd.DataFrame) -> Dict[str, Dict]:
    """Analyze class distribution and compute weights."""
    task_info = {
        'zone': {'col': 'zone_label', 'n_classes': 3, 'names': ['Zone I', 'Zone II', 'Zone III']},
        'stage': {'col': 'stage_label', 'n_classes': 4, 'names': ['Stage 0', 'Stage 1', 'Stage 2', 'Stage 3']},
        'plus': {'col': 'plus_label', 'n_classes': 3, 'names': ['Normal', 'PrePlus', 'Plus']},
        'aggressive_rop': {'col': 'aggressive_rop_label', 'n_classes': 2, 'names': ['No', 'Yes']},
        'treatment': {'col': 'treatment_label', 'n_classes': 2, 'names': ['No', 'Yes']},
    }
    
    class_weights = {}
    
    print("=" * 70)
    print("CLASS DISTRIBUTION AND WEIGHTS")
    print("=" * 70)
    
    for task, info in task_info.items():
        col = info['col']
        n_classes = info['n_classes']
        names = info['names']
        
        # Count valid samples per class
        valid_mask = df[col] >= 0
        counts = df[valid_mask][col].value_counts().sort_index()
        
        # Ensure all classes are present
        full_counts = [counts.get(i, 0) for i in range(n_classes)]
        total = sum(full_counts)
        
        # Compute weights: inverse frequency, normalized
        # weight_i = total / (n_classes * count_i)
        weights = []
        for c in full_counts:
            if c > 0:
                w = total / (n_classes * c)
            else:
                w = 1.0
            weights.append(w)
        
        # Normalize so min weight = 1.0
        min_w = min(weights)
        weights = [w / min_w for w in weights]
        
        class_weights[task] = torch.tensor(weights, dtype=torch.float32)
        
        print(f"\n{task.upper()}:")
        for i, (name, cnt, w) in enumerate(zip(names, full_counts, weights)):
            pct = cnt / total * 100 if total > 0 else 0
            print(f"  {name:12s}: {cnt:>5} ({pct:>5.1f}%)  weight: {w:.2f}")
    
    return class_weights

# Compute class weights from dataset
class_weights = analyze_class_distribution(dataset_df)

## 2. Dataset and Augmentation

In [5]:
# ==================== Augmentation ====================

def get_train_transforms(img_size: int = 512) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Rotate(limit=180, p=0.9, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.OneOf([
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1.0),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
        ], p=0.7),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=5, p=1.0),
        ], p=0.3),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.85, 1.0), ratio=(0.9, 1.1), p=0.4),
        A.CoarseDropout(max_holes=8, max_height=img_size//16, max_width=img_size//16, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_valid_transforms(img_size: int = 512) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

train_transforms = get_train_transforms(config.IMG_SIZE)
valid_transforms = get_valid_transforms(config.IMG_SIZE)

In [6]:
# ==================== Dataset ====================

class ROPDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transforms: A.Compose = None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.task_columns = {
            'zone': 'zone_label',
            'stage': 'stage_label',
            'plus': 'plus_label',
            'aggressive_rop': 'aggressive_rop_label',
            'treatment': 'treatment_label'
        }
    
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        
        image = cv2.imread(row['image_path'])
        if image is None:
            raise ValueError(f"Failed to load: {row['image_path']}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transforms:
            image = self.transforms(image=image)['image']
        
        labels = {}
        for task_name, col_name in self.task_columns.items():
            labels[task_name] = torch.tensor(int(row.get(col_name, -1)), dtype=torch.long)
        
        return {'image': image, 'labels': labels, 'image_path': row['image_path']}

## 3. Model

In [7]:
# ==================== Model ====================

class MultiTaskHead(nn.Module):
    def __init__(self, in_features: int, num_classes: int, dropout: float = 0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.head(x)


class ROPMultiTaskModel(nn.Module):
    TASK_CONFIG = {
        'zone': (3, True),
        'stage': (4, True),
        'plus': (3, True),
        'aggressive_rop': (2, False),
        'treatment': (2, False)
    }
    
    def __init__(self, model_name: str = "efficientnet_b0", pretrained: bool = True, dropout: float = 0.3):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        in_features = self.backbone.num_features
        
        self.heads = nn.ModuleDict()
        for task_name, (num_classes, _) in self.TASK_CONFIG.items():
            self.heads[task_name] = MultiTaskHead(in_features, num_classes, dropout)
    
    def forward(self, x):
        features = self.backbone(x)
        return {task: head(features) for task, head in self.heads.items()}

## 4. Loss Functions (Class-Balanced)

In [8]:
# ==================== Weighted Focal Loss ====================

class WeightedFocalLoss(nn.Module):
    """
    Focal Loss with class weights (alpha).
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        alpha: Class weights tensor [n_classes]
        gamma: Focusing parameter (default: 2.0)
    """
    
    def __init__(self, alpha: torch.Tensor = None, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        
        # Apply class weights
        if self.alpha is not None:
            alpha = self.alpha.to(inputs.device)
            alpha_t = alpha[targets]
            focal_loss = alpha_t * focal_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

In [9]:
# ==================== Multi-Task Loss (Class-Balanced) ====================

class ClassBalancedMultiTaskLoss(nn.Module):
    """
    Multi-task loss with class balancing.
    
    Step 1: Weighted Cross-Entropy (クラス頻度逆数)
    Step 2: Weighted Focal Loss (alpha + gamma)
    """
    
    def __init__(
        self,
        task_weights: Dict[str, float],
        class_weights: Dict[str, torch.Tensor],
        label_smoothing: float = 0.1,
        use_focal: bool = True,
        focal_gamma: float = 2.0
    ):
        super().__init__()
        self.task_weights = task_weights
        self.class_weights = class_weights
        self.label_smoothing = label_smoothing
        self.use_focal = use_focal
        self.focal_gamma = focal_gamma
        
        # Create focal loss for each task
        self.focal_losses = nn.ModuleDict()
        for task, weights in class_weights.items():
            self.focal_losses[task] = WeightedFocalLoss(alpha=weights, gamma=focal_gamma)
    
    def forward(self, outputs: Dict[str, torch.Tensor], labels: Dict[str, torch.Tensor]):
        total_loss = 0.0
        task_losses = {}
        
        for task_name, output in outputs.items():
            target = labels[task_name]
            mask = target >= 0
            
            if mask.sum() == 0:
                task_losses[task_name] = torch.tensor(0.0, device=output.device)
                continue
            
            output = output[mask]
            target = target[mask]
            
            # Get class weights for this task
            weights = self.class_weights.get(task_name)
            if weights is not None:
                weights = weights.to(output.device)
            
            # Weighted Cross-Entropy
            ce_loss = F.cross_entropy(
                output, target,
                weight=weights,
                label_smoothing=self.label_smoothing
            )
            
            # Optionally add Focal Loss
            if self.use_focal and task_name in self.focal_losses:
                focal_loss = self.focal_losses[task_name](output, target)
                loss = (ce_loss + focal_loss) / 2
            else:
                loss = ce_loss
            
            task_weight = self.task_weights.get(task_name, 1.0)
            task_losses[task_name] = loss
            total_loss += task_weight * loss
        
        return total_loss, task_losses

## 5. Training

In [10]:
# ==================== Metrics ====================

def compute_metrics(preds: Dict[str, np.ndarray], labels: Dict[str, np.ndarray]) -> Dict:
    metrics = {}
    
    for task_name in preds.keys():
        y_pred = preds[task_name]
        y_true = labels[task_name]
        
        mask = y_true >= 0
        y_pred = y_pred[mask]
        y_true = y_true[mask]
        
        if len(y_true) == 0:
            metrics[task_name] = {'accuracy': 0.0, 'kappa': 0.0, 'f1_macro': 0.0}
            continue
        
        pred_classes = y_pred.argmax(axis=1) if y_pred.ndim > 1 else y_pred
        
        task_metrics = {
            'accuracy': accuracy_score(y_true, pred_classes),
            'kappa': cohen_kappa_score(y_true, pred_classes, weights='quadratic'),
            'f1_macro': f1_score(y_true, pred_classes, average='macro', zero_division=0),
        }
        
        # Sensitivity for minority class (binary tasks)
        if task_name in ['aggressive_rop', 'treatment']:
            cm = confusion_matrix(y_true, pred_classes, labels=[0, 1])
            if cm.shape == (2, 2) and (cm[1, 0] + cm[1, 1]) > 0:
                task_metrics['sensitivity'] = cm[1, 1] / (cm[1, 0] + cm[1, 1])
            else:
                task_metrics['sensitivity'] = 0.0
            
            try:
                probs = torch.softmax(torch.tensor(y_pred), dim=1)[:, 1].numpy()
                task_metrics['auc'] = roc_auc_score(y_true, probs)
            except:
                task_metrics['auc'] = 0.0
        
        metrics[task_name] = task_metrics
    
    return metrics

In [11]:
# ==================== MixUp ====================

def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, {k: v[index] for k, v in y.items()}
    
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    loss_a, _ = criterion(pred, y_a)
    loss_b, _ = criterion(pred, y_b)
    return lam * loss_a + (1 - lam) * loss_b

In [12]:
# ==================== Training Loop ====================

def train_epoch(model, dataloader, criterion, optimizer, scheduler, device, mixup_alpha=0.2, mixup_p=0.5):
    model.train()
    running_loss = 0.0
    
    for batch in tqdm(dataloader, desc="Training", leave=False):
        images = batch['image'].to(device)
        labels = {k: v.to(device) for k, v in batch['labels'].items()}
        
        if np.random.random() < mixup_p:
            images, labels_a, labels_b, lam = mixup_data(images, labels, mixup_alpha)
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        else:
            outputs = model(images)
            loss, _ = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
    
    return running_loss / len(dataloader)


def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = {task: [] for task in ROPMultiTaskModel.TASK_CONFIG.keys()}
    all_labels = {task: [] for task in ROPMultiTaskModel.TASK_CONFIG.keys()}
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation", leave=False):
            images = batch['image'].to(device)
            labels = {k: v.to(device) for k, v in batch['labels'].items()}
            
            outputs = model(images)
            loss, _ = criterion(outputs, labels)
            running_loss += loss.item()
            
            for task in all_preds.keys():
                all_preds[task].append(outputs[task].cpu().numpy())
                all_labels[task].append(labels[task].cpu().numpy())
    
    for task in all_preds.keys():
        all_preds[task] = np.concatenate(all_preds[task])
        all_labels[task] = np.concatenate(all_labels[task])
    
    metrics = compute_metrics(all_preds, all_labels)
    
    return running_loss / len(dataloader), metrics, all_preds, all_labels

In [ ]:
# ==================== Cross-Validation ====================

def run_cross_validation(df: pd.DataFrame, class_weights: Dict, config: Config) -> Dict:
    """Run 5-fold cross-validation with class-balanced loss."""
    output_dir = config.OUTPUT_DIR
    output_dir.mkdir(parents=True, exist_ok=True)
    
    df['stratify_target'] = df['zone_label'].astype(str) + '_' + df['stage_label'].astype(str)
    sgkf = StratifiedGroupKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=42)
    
    all_fold_results = []
    all_predictions = []
    
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, df['stratify_target'], df['video_id'])):
        fold_dir = output_dir / f"fold_{fold + 1}"
        fold_dir.mkdir(exist_ok=True)
        best_model_path = fold_dir / "best_model.pt"
        
        # Skip if already trained
        if best_model_path.exists():
            print(f"\nFold {fold + 1}/{config.N_FOLDS} - SKIPPED (already trained)")
            
            val_df = df.iloc[val_idx].reset_index(drop=True)
            val_dataset = ROPDataset(val_df, transforms=valid_transforms)
            val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
            
            model = ROPMultiTaskModel(config.MODEL_NAME, pretrained=False, dropout=config.DROPOUT)
            model.load_state_dict(torch.load(best_model_path, map_location=device))
            model = model.to(device)
            
            criterion = ClassBalancedMultiTaskLoss(
                task_weights=config.TASK_WEIGHTS,
                class_weights=class_weights,
                label_smoothing=config.LABEL_SMOOTHING,
                use_focal=config.USE_FOCAL_LOSS,
                focal_gamma=config.FOCAL_GAMMA
            )
            
            val_loss, final_metrics, preds, labels = validate_epoch(model, val_loader, criterion, device)
            
            # Print key metrics
            print(f"  Aggressive ROP Sensitivity: {final_metrics['aggressive_rop'].get('sensitivity', 0):.4f}")
            print(f"  Treatment Sensitivity: {final_metrics['treatment'].get('sensitivity', 0):.4f}")
            
            all_fold_results.append({
                'fold': fold + 1,
                'val_size': len(val_idx),
                'best_val_loss': val_loss,
                'metrics': final_metrics
            })
            
            for i, row in val_df.iterrows():
                pred_entry = {'fold': fold + 1, 'image_path': row['image_path'], 'video_id': row['video_id']}
                idx = val_df.index.get_loc(i)
                for task in preds.keys():
                    pred_entry[f'{task}_label'] = labels[task][idx]
                    pred_entry[f'{task}_pred'] = preds[task][idx].argmax()
                    if preds[task][idx].ndim > 0:
                        pred_entry[f'{task}_prob'] = torch.softmax(torch.tensor(preds[task][idx]), dim=0)[-1].item()
                all_predictions.append(pred_entry)
            
            del model
            torch.cuda.empty_cache()
            continue
        
        # Train fold
        print(f"\n{'='*50}")
        print(f"Fold {fold + 1}/{config.N_FOLDS}")
        print(f"{'='*50}")
        
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)
        print(f"Train: {len(train_df)}, Val: {len(val_df)}")
        
        # Recalculate class weights from training data only
        fold_class_weights = {}
        for task, col in [('zone', 'zone_label'), ('stage', 'stage_label'), ('plus', 'plus_label'),
                          ('aggressive_rop', 'aggressive_rop_label'), ('treatment', 'treatment_label')]:
            valid_mask = train_df[col] >= 0
            counts = train_df[valid_mask][col].value_counts().sort_index()
            n_classes = class_weights[task].shape[0]
            full_counts = [counts.get(i, 1) for i in range(n_classes)]
            total = sum(full_counts)
            weights = [total / (n_classes * c) for c in full_counts]
            min_w = min(weights)
            weights = [w / min_w for w in weights]
            fold_class_weights[task] = torch.tensor(weights, dtype=torch.float32)
        
        print(f"Fold class weights (aggressive_rop): {fold_class_weights['aggressive_rop'].tolist()}")
        
        train_dataset = ROPDataset(train_df, transforms=train_transforms)
        val_dataset = ROPDataset(val_df, transforms=valid_transforms)
        
        train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
        
        model = ROPMultiTaskModel(config.MODEL_NAME, pretrained=config.PRETRAINED, dropout=config.DROPOUT)
        model = model.to(device)
        
        criterion = ClassBalancedMultiTaskLoss(
            task_weights=config.TASK_WEIGHTS,
            class_weights=fold_class_weights,
            label_smoothing=config.LABEL_SMOOTHING,
            use_focal=config.USE_FOCAL_LOSS,
            focal_gamma=config.FOCAL_GAMMA
        )
        
        optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
        scheduler = OneCycleLR(optimizer, max_lr=config.LEARNING_RATE, epochs=config.EPOCHS, steps_per_epoch=len(train_loader))
        
        best_val_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(config.EPOCHS):
            train_loss = train_epoch(model, train_loader, criterion, optimizer, scheduler, device, config.MIXUP_ALPHA, config.MIXUP_P)
            val_loss, metrics, _, _ = validate_epoch(model, val_loader, criterion, device)
            
            arop_sens = metrics['aggressive_rop'].get('sensitivity', 0)
            treat_sens = metrics['treatment'].get('sensitivity', 0)
            
            print(f"Epoch {epoch + 1}: Loss={val_loss:.4f}, AROP_Sens={arop_sens:.3f}, Treat_Sens={treat_sens:.3f}")
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), best_model_path)
            else:
                patience_counter += 1
                if patience_counter >= config.PATIENCE:
                    print(f"Early stopping at epoch {epoch + 1}")
                    break
        
        model.load_state_dict(torch.load(best_model_path))
        _, final_metrics, preds, labels = validate_epoch(model, val_loader, criterion, device)
        
        all_fold_results.append({
            'fold': fold + 1,
            'train_size': len(train_df),
            'val_size': len(val_df),
            'best_val_loss': best_val_loss,
            'metrics': final_metrics
        })
        
        for i, row in val_df.iterrows():
            pred_entry = {'fold': fold + 1, 'image_path': row['image_path'], 'video_id': row['video_id']}
            idx = val_df.index.get_loc(i)
            for task in preds.keys():
                pred_entry[f'{task}_label'] = labels[task][idx]
                pred_entry[f'{task}_pred'] = preds[task][idx].argmax()
                if preds[task][idx].ndim > 0:
                    pred_entry[f'{task}_prob'] = torch.softmax(torch.tensor(preds[task][idx]), dim=0)[-1].item()
            all_predictions.append(pred_entry)
        
        del model, optimizer, scheduler
        torch.cuda.empty_cache()
    
    results_df = pd.DataFrame(all_predictions)
    results_df.to_csv(output_dir / "predictions.csv", index=False)
    
    return {'fold_results': all_fold_results, 'predictions_df': results_df}

## 6. Run Experiment

In [ ]:
# ==================== Run Training ====================

print("Configuration:")
print(f"  USE_CLASS_WEIGHTS: {config.USE_CLASS_WEIGHTS}")
print(f"  USE_FOCAL_LOSS: {config.USE_FOCAL_LOSS}")
print(f"  FOCAL_GAMMA: {config.FOCAL_GAMMA}")
print(f"  LABEL_SMOOTHING: {config.LABEL_SMOOTHING}")
print()

results = run_cross_validation(dataset_df, class_weights, config)

Configuration:
  USE_CLASS_WEIGHTS: True
  USE_FOCAL_LOSS: True
  FOCAL_GAMMA: 2.0
  LABEL_SMOOTHING: 0.1


Fold 1/5
Train: 5116, Val: 1332
Fold class weights (aggressive_rop): [1.0, 23.834951400756836]


Training:   0%|          | 0/320 [00:00<?, ?it/s]

Validation:   0%|          | 0/84 [00:00<?, ?it/s]

Epoch 1: Loss=4.5824, AROP_Sens=0.955, Treat_Sens=0.295


Training:   0%|          | 0/320 [00:00<?, ?it/s]

...

Epoch 33: Loss=3.9679, AROP_Sens=0.500, Treat_Sens=0.864


Training:   0%|          | 0/320 [00:00<?, ?it/s]

Validation:   0%|          | 0/84 [00:00<?, ?it/s]

Epoch 34: Loss=3.9727, AROP_Sens=0.136, Treat_Sens=0.784


Training:   0%|          | 0/320 [00:00<?, ?it/s]

## 7. Results

In [ ]:
# ==================== Aggregate Results ====================

def aggregate_metrics(fold_results):
    agg = {}
    tasks = fold_results[0]['metrics'].keys()
    for task in tasks:
        task_agg = {}
        for metric in fold_results[0]['metrics'][task].keys():
            values = [f['metrics'][task][metric] for f in fold_results]
            task_agg[metric] = (np.mean(values), np.std(values))
        agg[task] = task_agg
    return agg

agg = aggregate_metrics(results['fold_results'])

print("\n" + "=" * 70)
print("CLASS-BALANCED TRAINING RESULTS")
print("=" * 70)

for task in ['zone', 'stage', 'plus', 'aggressive_rop', 'treatment']:
    print(f"\n{task.upper()}:")
    for metric, (mean, std) in agg[task].items():
        print(f"  {metric:15s}: {mean:.4f} ± {std:.4f}")

In [ ]:
# ==================== Compare with Baseline ====================

# Baseline results (from good_fair without class balancing)
baseline = {
    'zone': {'accuracy': 0.7137, 'kappa': 0.5024},
    'stage': {'accuracy': 0.6531, 'kappa': 0.6674},
    'aggressive_rop': {'accuracy': 0.9681, 'sensitivity': 0.447, 'auc': 0.7202},
    'treatment': {'accuracy': 0.9413, 'sensitivity': 0.654, 'auc': 0.8184},
}

print("\n" + "=" * 70)
print("COMPARISON: Baseline vs Class-Balanced")
print("=" * 70)

comparison_rows = []
for task in ['zone', 'stage', 'plus', 'aggressive_rop', 'treatment']:
    if task not in baseline:
        # No baseline available for this task (Plus was not evaluated in baseline)
        for metric, (mean, std) in agg[task].items():
            comparison_rows.append({
                'Task': task,
                'Metric': metric,
                'Baseline': 'N/A',
                'Class-Balanced': f"{mean:.4f} ± {std:.4f}",
                'Diff': 'N/A'
            })
        continue
    for metric in baseline[task].keys():
        if metric in agg[task]:
            base_val = baseline[task][metric]
            new_mean, new_std = agg[task][metric]
            diff = new_mean - base_val
            comparison_rows.append({
                'Task': task,
                'Metric': metric,
                'Baseline': f"{base_val:.4f}",
                'Class-Balanced': f"{new_mean:.4f} ± {new_std:.4f}",
                'Diff': f"{diff:+.4f}"
            })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

In [ ]:
# ==================== Detailed Per-Class Analysis ====================

pred_df = results['predictions_df']

task_class_names = {
    'zone': {0: 'Zone I', 1: 'Zone II', 2: 'Zone III'},
    'stage': {0: 'Stage 0', 1: 'Stage 1', 2: 'Stage 2', 3: 'Stage 3'},
    'plus': {0: 'Normal', 1: 'PrePlus', 2: 'Plus'},
    'aggressive_rop': {0: 'No', 1: 'Yes'},
    'treatment': {0: 'No', 1: 'Yes'},
}

for task, class_map in task_class_names.items():
    y_true = pred_df[f'{task}_label'].values.astype(int)
    y_pred = pred_df[f'{task}_pred'].values.astype(int)
    
    mask = y_true >= 0
    y_true, y_pred = y_true[mask], y_pred[mask]
    
    if len(y_true) == 0:
        continue
    
    classes = sorted(class_map.keys())
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    
    print(f"\n{'='*50}")
    print(f"{task.upper()}")
    print(f"{'='*50}")
    
    rows = []
    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        n = int(cm[i, :].sum())
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        rows.append({'Class': class_map[cls], 'n': n, 'Sensitivity': f"{sens:.3f}", 'PPV': f"{ppv:.3f}"})
    
    display(pd.DataFrame(rows))

In [ ]:
# ==================== Confusion Matrix Visualization ====================

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, (task, class_map) in enumerate(task_class_names.items()):
    ax = axes[idx]
    y_true = pred_df[f'{task}_label'].values.astype(int)
    y_pred = pred_df[f'{task}_pred'].values.astype(int)
    
    mask = y_true >= 0
    y_true, y_pred = y_true[mask], y_pred[mask]
    
    classes = sorted(class_map.keys())
    labels = [class_map[c] for c in classes]
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    
    # Normalize by row (true label) to show recall per class
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    # Show both count and percentage
    annot = np.array([[f"{cnt}\n({pct:.1%})" for cnt, pct in zip(row_cnt, row_pct)]
                      for row_cnt, row_pct in zip(cm, cm_norm)])
    
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', vmin=0, vmax=1,
                xticklabels=labels, yticklabels=labels, ax=ax, cbar_kws={'shrink': 0.8})
    ax.set_title(f'{task.upper()}', fontsize=14, fontweight='bold')
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')

# Hide unused subplot
axes[5].set_visible(False)

fig.suptitle('Confusion Matrices (Row-Normalized)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to: {config.OUTPUT_DIR / 'confusion_matrices.png'}")

In [ ]:
# ==================== Save Configuration ====================

config_dict = {
    'version': config.VERSION,
    'quality_filter': config.QUALITY_FILTER,
    'use_class_weights': config.USE_CLASS_WEIGHTS,
    'use_focal_loss': config.USE_FOCAL_LOSS,
    'focal_gamma': config.FOCAL_GAMMA,
    'label_smoothing': config.LABEL_SMOOTHING,
    'class_weights': {k: v.tolist() for k, v in class_weights.items()},
    'results': {
        task: {metric: f"{mean:.4f} ± {std:.4f}" for metric, (mean, std) in metrics.items()}
        for task, metrics in agg.items()
    }
}

import json
with open(config.OUTPUT_DIR / 'config.json', 'w') as f:
    json.dump(config_dict, f, indent=2)

print(f"Configuration saved to: {config.OUTPUT_DIR / 'config.json'}")

Configuration saved to: C:\Users\ykita\ROP_AI_project\ROP_project\multicenter_study\outputs_class_balanced\config.json
